# 07. Tree Models — Stage 0

This notebook investigates tree-based models for the Stage-0 volatility
prediction problem.

The objective is to determine whether nonlinear tree-based models can capture
relationships between the statistical and topological features constructed in
the previous notebooks that are not captured by the logistic-regression
baseline.

The models will be evaluated using the same chronological train, validation,
and test framework established in `06_baseline_model.ipynb`.

The primary models considered in this notebook are:

- Random Forest
- XGBoost

Hyperparameter experiments will be performed using the training data, while
the final test set will remain untouched until the final evaluation.

Unlike the logistic-regression baseline, tree-based models do not require
feature standardization.

## 1. Project Setup and Imports

### 1.1: Define Project Path

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

STAGE0_DATA_FILE = (
    PROCESSED_DATA_DIR / "spy_stage0_features.csv"
)

### 1.2: Import Libraries

In [2]:
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt
import xgboost as xgb
# project modules
from src.evaluation import evaluate_classification_model, classification_metrics
from src.models import create_random_forest

## 2. Load Stage 0 Dataset

In [3]:
# Load the Stage-0 dataset.

stage0_data = pd.read_csv(
    STAGE0_DATA_FILE,
    index_col=0,
    parse_dates=True
)

stage0_data.head()

,log_return,abs_return,squared_return,vol_5d,vol_10d,vol_20d,vol_60d,vol_ratio_5_20,vol_ratio_20_60,abs_return_lag_1,...,h0_mean_persistence,h0_max_persistence,h1_total_persistence,h1_mean_persistence,h1_max_persistence,h1_n_features,h1_l1_norm,h1_l2_norm,h1_sup_norm,target
date,,,,,,,,,,,,,,,,,,,,,
2010-03-31,-0.003413,0.003413,0.000012,0.003322,0.004271,0.005152,0.008848,0.644720,0.582289,0.000681,...,0.003322,0.007881,0.000868,0.000868,0.000868,1,1.883799e-07,0.000007,0.000434,0
2010-04-01,0.006815,0.006815,0.000046,0.004449,0.004781,0.005330,0.008885,0.834670,0.599885,0.003413,...,0.002969,0.007881,0.000868,0.000868,0.000868,1,1.883799e-07,0.000007,0.000434,0
2010-04-05,0.008116,0.008116,0.000066,0.005735,0.005184,0.004651,0.008946,1.233187,0.519877,0.006815,...,0.002615,0.004491,0.000868,0.000868,0.000868,1,1.883799e-07,0.000007,0.000434,0
2010-04-06,0.002355,0.002355,0.000006,0.005098,0.004958,0.004680,0.008934,1.089330,0.523846,0.008116,...,0.002702,0.004491,0.000688,0.000344,0.000555,2,8.148190e-08,0.000004,0.000278,0
2010-04-07,-0.005729,0.005729,0.000033,0.005698,0.004791,0.004838,0.008955,1.177664,0.540292,0.002355,...,0.002828,0.005156,0.001775,0.000444,0.001358,4,4.757887e-07,0.000014,0.000679,1


## 3. Chronological Train, Validation, and Test Split

We use the same chronological split established in
`06_baseline_model.ipynb` so that the Stage-0 models can be compared under
the same evaluation framework.

The data are divided into:

- **Training:** 2010--2020
- **Validation:** 2021--2022
- **Test:** 2023--2026

The chronological ordering is preserved throughout. No observations are
randomly shuffled between the three sets.

The validation set will be used during model development and
hyperparameter experiments. The test set will remain untouched until the
final evaluation.

In [4]:
TRAIN_END = "2020-12-31"
VALIDATION_END = "2022-12-31"

train_mask = stage0_data.index <= TRAIN_END

validation_mask = (
    (stage0_data.index > TRAIN_END)
    & (stage0_data.index <= VALIDATION_END)
)

test_mask = stage0_data.index > VALIDATION_END

train_data = stage0_data.loc[train_mask].copy()
validation_data = stage0_data.loc[validation_mask].copy()
test_data = stage0_data.loc[test_mask].copy()

In [5]:
FEATURE_COLUMNS = [
    column for column in stage0_data.columns
    if column != "target"
]

X_train = train_data[FEATURE_COLUMNS]
y_train = train_data["target"]

X_validation = validation_data[FEATURE_COLUMNS]
y_validation = validation_data["target"]

X_test = test_data[FEATURE_COLUMNS]
y_test = test_data["target"]

In [6]:
# verify
print("Training:", X_train.shape, y_train.shape)
print("Validation:", X_validation.shape, y_validation.shape)
print("Test:", X_test.shape, y_test.shape)

Training: (2709, 27) (2709,)
Validation: (503, 27) (503,)
Test: (919, 27) (919,)


## 4. Random Forest

### 4.1 Model Construction

Random Forest is an ensemble method that combines predictions from many
decision trees.

Each tree is trained using a randomized procedure, and the individual tree
predictions are aggregated to produce the final classification.

Random Forests can capture nonlinear relationships and interactions between
features without requiring explicit feature transformations.

For the first experiment, we use a baseline set of hyperparameters. These
parameters will provide a reference point before we investigate
hyperparameter variation.

In [7]:
# construct the model 
random_forest = create_random_forest()

# inspect it
random_forest

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",200
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total

### 4.2: Verify and Fit Model

In [8]:
# verify the constructor -> False 
hasattr(random_forest, "estimators_")

False

In [9]:
# fit the model 
random_forest.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",200
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total

In [10]:
# verify that fitting occurred -> True
hasattr(random_forest, "estimators_")

True

### 4.3: Baseline Random Forest

In [11]:
# generate validation prediction
y_validation_pred = random_forest.predict(X_validation)
y_validation_proba = random_forest.predict_proba(
    X_validation
)[:, 1]

In [12]:
random_forest_validation_metrics = classification_metrics(
    y_validation,
    y_validation_pred,
    y_validation_proba,
)

random_forest_validation_metrics

{'accuracy': 0.7614314115308151,
 'balanced_accuracy': 0.5370194939160456,
 'precision': 0.6875,
 'recall': 0.0873015873015873,
 'f1': 0.15492957746478872,
 'roc_auc': 0.670877015704602,
 'pr_auc': 0.4335443774561032,
 'log_loss': 0.5513239160475043}

**Next: inspect the predictions**

Before tuning, we should establish one useful diagnostic: how many positive predictions is the Random Forest actually making?

In [13]:
pd.Series(y_validation_pred).value_counts()

0    487
1     16
Name: count, dtype: int64

In [14]:
pd.Series(y_validation_proba).describe()

count    503.000000
mean       0.202087
std        0.141570
min        0.010000
25%        0.082500
50%        0.165000
75%        0.300000
max        0.640000
dtype: float64

### 4.4 Classification Threshold Analysis

The Random Forest produces relatively few positive predictions at the default
classification threshold of 0.5.

Since the positive class represents approximately one quarter of the
validation observations, we examine how the classification metrics change as
the probability threshold varies.

The model itself is kept fixed in this analysis. Only the threshold used to
convert predicted probabilities into binary predictions is changed.

This allows us to distinguish the model's ability to rank observations from
the effect of the particular classification threshold.

In [15]:
thresholds = [0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5]

threshold_results = []

for threshold in thresholds:
    y_pred_threshold = (
        y_validation_proba >= threshold
    ).astype(int)

    metrics = classification_metrics(
        y_validation,
        y_pred_threshold,
        y_validation_proba,
    )

    metrics["threshold"] = threshold
    threshold_results.append(metrics)

threshold_results = pd.DataFrame(threshold_results)

threshold_results

,accuracy,balanced_accuracy,precision,recall,f1,roc_auc,pr_auc,log_loss,threshold
0,0.630219,0.623858,0.359813,0.611111,0.452941,0.670877,0.433544,0.551324,0.20
1,0.683897,0.643815,0.405714,0.563492,0.471761,0.670877,0.433544,0.551324,0.25
2,0.719682,0.630700,0.441860,0.452381,0.447059,0.670877,0.433544,0.551324,0.30
3,0.749503,0.621532,0.500000,0.365079,0.422018,0.670877,0.433544,0.551324,0.35
4,0.741551,0.566029,0.465517,0.214286,0.293478,0.670877,0.433544,0.551324,0.40
5,0.761431,0.563439,0.583333,0.166667,0.259259,0.670877,0.433544,0.551324,0.45
6,0.761431,0.537019,0.687500,0.087302,0.154930,0.670877,0.433544,0.551324,0.50


**Observation:** The default threshold of 0.5 was making the forest extremely conservative: threshold = 0.50 → recall = 0.087. The underlying probability ranking contains more information than that binary prediction suggests. Changing the threshold exposes substantially more of that information.

### 4.5 Selected Classification Threshold

For the threshold analysis, we select the threshold that maximizes the F1
score on the validation set among the candidate thresholds considered above.

This threshold is selected using the validation data only. The test set is not
used in selecting the threshold.

The selected threshold will be retained as a candidate decision threshold for
later evaluation.

In [16]:
best_threshold = threshold_results.loc[
    threshold_results["f1"].idxmax(),
    "threshold",
]

best_threshold

np.float64(0.25)

In [17]:
best_threshold_metrics = threshold_results.loc[
    threshold_results["f1"].idxmax()
].to_dict()

best_threshold_metrics

{'accuracy': 0.68389662027833,
 'balanced_accuracy': 0.6438149972632732,
 'precision': 0.4057142857142857,
 'recall': 0.5634920634920635,
 'f1': 0.4717607973421927,
 'roc_auc': 0.670877015704602,
 'pr_auc': 0.4335443774561032,
 'log_loss': 0.5513239160475043,
 'threshold': 0.25}

## 5. Hyperparameter Tuning Experiments

We now investigate whether the performance of the Random Forest can be
improved by varying its hyperparameters.

Hyperparameters are selected using time-series cross-validation within the
training period only. The validation and test periods are not used during
this experiment.

Because the positive class is relatively uncommon, we focus primarily on
Average Precision (PR-AUC) and ROC-AUC when comparing model configurations.
These metrics evaluate the quality of the probability ranking without
requiring us to choose a classification threshold during hyperparameter
selection.

We begin with tree-complexity parameters while keeping the other parameters
fixed. This gives us a controlled experiment rather than a large
hyperparameter search.

### 5.1: Time-Series Cross-Validation

In [18]:
from sklearn.model_selection import TimeSeriesSplit

tscv = TimeSeriesSplit(n_splits=5)

### 5.2: First Hyperparameter Experiment:
- max_depth
- min_samples_leaf

In [19]:
max_depth_values = [5, 10, None]
min_samples_leaf_values = [1, 5, 10]

In [20]:
hyperparameter_results = []

for max_depth in max_depth_values:
    for min_samples_leaf in min_samples_leaf_values:

        model = create_random_forest(
            n_estimators=200,
            max_depth=max_depth,
            min_samples_leaf=min_samples_leaf,
            max_features="sqrt",
            random_state=42,
            n_jobs=-1,
        )

        fold_results = evaluate_classification_model(
            model,
            X_train,
            y_train,
            tscv,
        )

        hyperparameter_results.append(
            {
                "max_depth": max_depth,
                "min_samples_leaf": min_samples_leaf,
                "roc_auc_mean": fold_results["roc_auc"].mean(),
                "roc_auc_std": fold_results["roc_auc"].std(),
                "pr_auc_mean": fold_results["pr_auc"].mean(),
                "pr_auc_std": fold_results["pr_auc"].std(),
            }
        )

hyperparameter_results = pd.DataFrame(
    hyperparameter_results
)

hyperparameter_results

,max_depth,min_samples_leaf,roc_auc_mean,roc_auc_std,pr_auc_mean,pr_auc_std
0,5.0,1,0.681702,0.063975,0.416168,0.103761
1,5.0,5,0.685558,0.066233,0.407513,0.099682
2,5.0,10,0.680758,0.066151,0.404614,0.094605
3,10.0,1,0.667248,0.068059,0.396495,0.100018
4,10.0,5,0.671784,0.066844,0.395835,0.099004
5,10.0,10,0.681066,0.067788,0.405912,0.097816
6,NaN,1,0.663661,0.055928,0.386878,0.086988
7,NaN,5,0.673181,0.067183,0.395144,0.097159
8,NaN,10,0.679838,0.065994,0.405588,0.096777


The CV experiment suggests that unrestricted trees are not obviously advantageous for this problem. Shallower trees appear to generalize better within the training-period CV folds.

### 5.3 Number of Trees and Feature Subsampling

The first experiment suggested that relatively shallow trees provide a useful
degree of regularization for this problem.

We now fix the tree-complexity parameters at

$$
\text{max\_depth}=5,
\qquad
\text{min\_samples\_leaf}=1,
$$

and investigate two additional Random Forest hyperparameters:

- `n_estimators`: the number of trees in the forest;
- `max_features`: the number of features considered when constructing each
  split.

We consider

$$
n_{\mathrm{estimators}}\in\{100,200,500\}
$$

and

$$
\text{max\_features}\in
\{\text{"sqrt"},\text{"log2"},\text{None}\}.
$$

The models are again evaluated using five-fold time-series cross-validation
within the training period. The validation and test periods remain untouched.

In [21]:
n_estimators_values = [100, 200, 500]
max_features_values = ["sqrt", "log2", None]

tree_subsampling_results = []

for n_estimators in n_estimators_values:
    for max_features in max_features_values:

        model = create_random_forest(
            n_estimators=n_estimators,
            max_depth=5,
            min_samples_leaf=1,
            max_features=max_features,
            random_state=42,
            n_jobs=-1,
        )

        fold_results = evaluate_classification_model(
            model,
            X_train,
            y_train,
            tscv,
        )

        tree_subsampling_results.append(
            {
                "n_estimators": n_estimators,
                "max_features": max_features,
                "roc_auc_mean": fold_results["roc_auc"].mean(),
                "roc_auc_std": fold_results["roc_auc"].std(),
                "pr_auc_mean": fold_results["pr_auc"].mean(),
                "pr_auc_std": fold_results["pr_auc"].std(),
            }
        )

tree_subsampling_results = pd.DataFrame(
    tree_subsampling_results
)

tree_subsampling_results

,n_estimators,max_features,roc_auc_mean,roc_auc_std,pr_auc_mean,pr_auc_std
0,100,sqrt,0.678754,0.066779,0.408691,0.099110
1,100,log2,0.677242,0.052508,0.408022,0.098246
2,100,NaN,0.676550,0.075775,0.401650,0.097898
3,200,sqrt,0.681702,0.063975,0.416168,0.103761
4,200,log2,0.681003,0.054641,0.415102,0.102865
5,200,NaN,0.677183,0.077953,0.400279,0.101944
6,500,sqrt,0.682909,0.061927,0.411178,0.099924
7,500,log2,0.682393,0.056373,0.413700,0.101732
8,500,NaN,0.672811,0.080895,0.396021,0.104130


1. max_features="sqrt" is consistently strong
Across all three values of n_estimators, "sqrt" has the highest PR-AUC.
Using all features at every split (None) performs noticeably worse.
2. More trees don't produce a major improvement
Going from 100 → 200 trees improves PR-AUC:
$$
0.409 \rightarrow 0.416
$$
but going from 200 → 500 gives:
$$
0.416 \rightarrow 0.411.
$$
So there isn't evidence here that 500 trees materially improves the probability ranking.

### 5.4 Selected Random Forest Configuration

The hyperparameter experiments were conducted using five-fold time-series
cross-validation within the training period.

Average Precision (PR-AUC) is used as the primary selection criterion because
the positive class is relatively uncommon. ROC-AUC is reported as a secondary
criterion.

Among the configurations examined, the following configuration produced the
highest mean PR-AUC:

$$
\text{n\_estimators}=200,
\qquad
\text{max\_depth}=5,
$$

$$
\text{min\_samples\_leaf}=1,
\qquad
\text{max\_features}=\text{"sqrt"}.
$$

This configuration will now be evaluated on the fixed 2021--2022 validation
period. The test period remains untouched.

In [22]:
selected_random_forest = create_random_forest(
    n_estimators=200,
    max_depth=5,
    min_samples_leaf=1,
    max_features="sqrt",
    random_state=42,
    n_jobs=-1,
)

In [23]:
selected_random_forest.fit(
    X_train,
    y_train,
)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",200
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",5
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total nu

In [24]:
y_validation_proba_rf = selected_random_forest.predict_proba(
    X_validation
)[:, 1]

#### 5.4.1 Candidate Classification Threshold

The selected Random Forest configuration is now evaluated on the fixed
validation period.

As in the baseline experiment, we examine a small set of candidate
classification thresholds. The model and its predicted probabilities remain
fixed; only the threshold used to convert probabilities into binary
predictions is varied.

The threshold is selected using the validation data and will not be
re-estimated using the final test set.

In [25]:
y_validation_proba_selected = selected_random_forest.predict_proba(
    X_validation
)[:, 1]

In [26]:
thresholds = [0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5]

selected_threshold_results = []

for threshold in thresholds:
    y_pred_threshold = (
        y_validation_proba_selected >= threshold
    ).astype(int)

    metrics = classification_metrics(
        y_validation,
        y_pred_threshold,
        y_validation_proba_selected,
    )

    metrics["threshold"] = threshold
    selected_threshold_results.append(metrics)

selected_threshold_results = pd.DataFrame(
    selected_threshold_results
)

selected_threshold_results

,accuracy,balanced_accuracy,precision,recall,f1,roc_auc,pr_auc,log_loss,threshold
0,0.397614,0.545303,0.272494,0.841270,0.411650,0.660898,0.426029,0.541461,0.10
1,0.550696,0.586660,0.312030,0.658730,0.423469,0.660898,0.426029,0.541461,0.15
2,0.689861,0.653078,0.414773,0.579365,0.483444,0.660898,0.426029,0.541461,0.20
3,0.721670,0.650520,0.450704,0.507937,0.477612,0.660898,0.426029,0.541461,0.25
4,0.745527,0.621521,0.489583,0.373016,0.423423,0.660898,0.426029,0.541461,0.30
5,0.773360,0.605743,0.607143,0.269841,0.373626,0.660898,0.426029,0.541461,0.35
6,0.757455,0.550219,0.566667,0.134921,0.217949,0.660898,0.426029,0.541461,0.40
7,0.751491,0.514536,0.555556,0.039683,0.074074,0.660898,0.426029,0.541461,0.45
8,0.753479,0.507937,1.000000,0.015873,0.031250,0.660898,0.426029,0.541461,0.50


In [27]:
best_threshold_selected_rf = selected_threshold_results.loc[
    selected_threshold_results["f1"].idxmax(),
    "threshold",
]

best_threshold_selected_rf

np.float64(0.2)

In [28]:
best_threshold_selected_rf_metrics = selected_threshold_results.loc[
    selected_threshold_results["f1"].idxmax()
].to_dict()

best_threshold_selected_rf_metrics

{'accuracy': 0.6898608349900597,
 'balanced_accuracy': 0.6530777651467308,
 'precision': 0.4147727272727273,
 'recall': 0.5793650793650794,
 'f1': 0.48344370860927155,
 'roc_auc': 0.6608984884846955,
 'pr_auc': 0.4260286998450852,
 'log_loss': 0.5414610619419116,
 'threshold': 0.2}

The selected candidate threshold is 0.20, which maximizes F1 among the candidate thresholds examined.

### 5.5 Validation Evaluation of Selected Random Forest

The Random Forest hyperparameters have been selected using time-series
cross-validation on the training period.

The classification threshold has subsequently been selected using the fixed
validation period, with F1 as the selection criterion among the candidate
thresholds examined.

We now evaluate the selected model on the validation period using the selected
threshold of `0.20`.

No further model or threshold selection is performed in this section.

In [29]:
y_validation_pred_selected = (
    y_validation_proba_selected >= best_threshold_selected_rf
).astype(int)

In [30]:
selected_rf_validation_metrics = classification_metrics(
    y_validation,
    y_validation_pred_selected,
    y_validation_proba_selected,
)

selected_rf_validation_metrics

{'accuracy': 0.6898608349900597,
 'balanced_accuracy': 0.6530777651467308,
 'precision': 0.4147727272727273,
 'recall': 0.5793650793650794,
 'f1': 0.48344370860927155,
 'roc_auc': 0.6608984884846955,
 'pr_auc': 0.4260286998450852,
 'log_loss': 0.5414610619419116}

### 5.6 Final Test Evaluation

The Random Forest hyperparameters were selected using time-series
cross-validation on the training period, and the classification threshold was
selected using the validation period.

We now freeze both choices:

$$
\text{n\_estimators}=200,\qquad
\text{max\_depth}=5,
$$

$$
\text{min\_samples\_leaf}=1,\qquad
\text{max\_features}=\text{"sqrt"},
$$

with classification threshold

$$
\theta = 0.20.
$$

Since the validation period has now served its purpose for model development,
the training and validation observations are combined to form the final
development dataset.

The final Random Forest is refitted on this combined 2010--2022 dataset and
then evaluated once on the untouched 2023--2026 test period.

The test set is not used for any model or threshold selection.

#### 5.6.1 Combine the Development Data

In [31]:
development_data = pd.concat(
    [
        train_data,
        validation_data,
    ]
)

In [32]:
X_development = development_data[FEATURE_COLUMNS]
y_development = development_data["target"]

print(
    "Development:",
    X_development.shape,
    y_development.shape,
)

print(
    "Test:",
    X_test.shape,
    y_test.shape,
)

Development: (3212, 27) (3212,)
Test: (919, 27) (919,)


#### 5.6.2: Refit the Selected Random Forest

In [33]:
final_random_forest = create_random_forest(
    n_estimators=200,
    max_depth=5,
    min_samples_leaf=1,
    max_features="sqrt",
    random_state=42,
    n_jobs=-1,
)

In [34]:
# fit on the combined development data
final_random_forest.fit(
    X_development,
    y_development,
)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",200
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",5
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total nu

#### 5.6.3: Generate Test Predictions

In [35]:
# get test probabilities
y_test_proba_rf = final_random_forest.predict_proba(
    X_test
)[:, 1]

In [36]:
# apply the frozen threshold of 0.2
y_test_pred_rf = (
    y_test_proba_rf >= best_threshold_selected_rf
).astype(int)

In [37]:
# evaluate 
final_rf_test_metrics = classification_metrics(
    y_test,
    y_test_pred_rf,
    y_test_proba_rf,
)

final_rf_test_metrics

{'accuracy': 0.588683351468988,
 'balanced_accuracy': 0.6765323803335499,
 'precision': 0.31274900398406374,
 'recall': 0.8263157894736842,
 'f1': 0.45375722543352603,
 'roc_auc': 0.7556927297668037,
 'pr_auc': 0.42165129212246877,
 'log_loss': 0.45376255956343686}

get confusion matrix:

In [38]:
from sklearn.metrics import confusion_matrix

test_confusion_matrix = confusion_matrix(
    y_test,
    y_test_pred_rf,
)

test_confusion_matrix

array([[384, 345],
       [ 33, 157]])

In [39]:
pd.Series(y_test_pred_rf).value_counts()

1    502
0    417
Name: count, dtype: int64

### 5.7 Random Forest Conclusions

The Random Forest experiments investigated both baseline performance and
controlled hyperparameter variation.

The hyperparameter experiments used five-fold time-series cross-validation
within the 2010--2020 training period. The selected configuration was

$$
\text{n\_estimators}=200,\qquad
\text{max\_depth}=5,
$$

$$
\text{min\_samples\_leaf}=1,\qquad
\text{max\_features}=\text{"sqrt"}.
$$

The classification threshold was subsequently selected on the 2021--2022
validation period. Among the candidate thresholds examined, a threshold of
0.20 produced the highest F1 score.

The final model was then refitted using the combined 2010--2022 development
data and evaluated on the untouched 2023--2026 test period.

At the selected threshold, the final test performance was:

- Accuracy: approximately 0.589
- Balanced accuracy: approximately 0.677
- Precision: approximately 0.313
- Recall: approximately 0.826
- F1: approximately 0.454
- ROC-AUC: approximately 0.757
- PR-AUC: approximately 0.422
- Log loss: approximately 0.454

The confusion matrix shows 157 true positives and 33 false negatives, but also
345 false positives. Thus, the high recall is accompanied by substantially
lower precision.

The test PR-AUC is very close to the validation PR-AUC, while ROC-AUC is
higher on the test period. These results should be interpreted as performance
on different historical market periods rather than as evidence of a temporal
improvement in the model.

The Random Forest therefore provides a useful nonlinear Stage-0 model and a
reference point for the subsequent XGBoost experiment.

## 6. XGBoost

XGBoost is a gradient-boosted tree model that builds an ensemble
sequentially, with each new tree attempting to improve the predictions of
the preceding ensemble.

Unlike Random Forest, where trees are constructed independently and their
predictions are aggregated, gradient boosting builds the model iteratively.

We investigate XGBoost as a second nonlinear Stage-0 model and compare its
performance with the logistic-regression and Random Forest baselines.

As with the Random Forest experiments, model development will preserve the
chronological structure of the financial time series. Hyperparameters will be
selected using time-series cross-validation within the training period, while
the validation and test periods will remain separated from the corresponding
selection procedures.

In [40]:
xgb.__version__

'3.2.0'

In [41]:
import importlib
import src.models

importlib.reload(src.models)

<module 'src.models' from '/Users/amitkumar/Desktop/Computing/Projects/Finance/market-volatility/src/models.py'>

In [42]:
from src.models import create_xgboost

### 6.1: Baseline XGBoost Model

In [43]:
xgboost_model = create_xgboost()

In [44]:
xgboost_model

,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",'logloss'
,feature_types feature_types: typing.Optional[typing.Sequence[str]].. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [45]:
# verify that it is initially unfitted -> False
hasattr(xgboost_model, "booster_")

False

In [46]:
# fit on the training period only 
xgboost_model.fit(
    X_train,
    y_train,
)

,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",'logloss'
,feature_types feature_types: typing.Optional[typing.Sequence[str]].. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [47]:
# verify the fitting -> True
hasattr(xgboost_model, "n_features_in_")

True

In [48]:
xgboost_model.get_booster().num_boosted_rounds()

200

### 6.2 Baseline XGBoost Evaluation

The baseline XGBoost model is now evaluated on the fixed validation period
(2021--2022).

As with the Random Forest, we evaluate both probability-ranking performance
and classification performance at the default threshold of 0.5.

No hyperparameter tuning is performed at this stage.

In [49]:
y_validation_pred_xgb = xgboost_model.predict(
    X_validation
)

y_validation_proba_xgb = xgboost_model.predict_proba(
    X_validation
)[:, 1]

In [50]:
xgb_validation_metrics = classification_metrics(
    y_validation,
    y_validation_pred_xgb,
    y_validation_proba_xgb,
)

xgb_validation_metrics

{'accuracy': 0.757455268389662,
 'balanced_accuracy': 0.5581449202138857,
 'precision': 0.5555555555555556,
 'recall': 0.15873015873015872,
 'f1': 0.24691358024691357,
 'roc_auc': 0.6873605321881182,
 'pr_auc': 0.44430800057705,
 'log_loss': 0.5440510511398315}

The comparison is interesting:

- XGBoost has better ROC-AUC and PR-AUC than the baseline Random Forest.
- It also has substantially better recall at the default 0.5 threshold.
- Its precision is lower, so it is less conservative about predicting Class 1.
- Its F1 is consequently better: 0.247 vs. 0.154.
- Accuracy is slightly lower, which isn't particularly important given the class imbalance.
  
So the baseline XGBoost looks more promising than the baseline Random Forest as a probability-ranking model.

**But the same threshold issue appears.**
The recall is still only:
$$
0.159
$$
at threshold 0.5.
Therefore, just as we did for Random Forest, we should inspect the predicted probabilities before tuning XGBoost.

In [51]:
pd.Series(y_validation_pred_xgb).value_counts()

0    467
1     36
Name: count, dtype: int64

In [52]:
pd.Series(y_validation_proba_xgb).describe()

count    503.000000
mean       0.197830
std        0.165413
min        0.012273
25%        0.069695
50%        0.141208
75%        0.288302
max        0.840230
dtype: float64

This is quite similar qualitatively to the Random Forest, whose median probability was 0.165 and whose 75th percentile was 0.300.
However, XGBoost has a noticeably higher maximum probability (0.840 vs. 0.640), and it produces more positive predictions at 0.5.
Next: threshold analysis
Let's use the same threshold-selection procedure we used for Random Forest so that the models remain comparable.

### 6.3 Classification Threshold Analysis

The baseline XGBoost model produces relatively few positive predictions at the
default threshold of 0.5.

We therefore examine a range of candidate thresholds while keeping the fitted
model and its predicted probabilities fixed.

As in the Random Forest experiment, F1 is used as the selection criterion,
with the threshold selected using the validation period only.

In [53]:
thresholds_xgb = [
    0.10,
    0.15,
    0.20,
    0.25,
    0.30,
    0.35,
    0.40,
    0.45,
    0.50,
]

xgb_threshold_results = []

for threshold in thresholds_xgb:
    y_pred_threshold = (
        y_validation_proba_xgb >= threshold
    ).astype(int)

    metrics = classification_metrics(
        y_validation,
        y_pred_threshold,
        y_validation_proba_xgb,
    )

    metrics["threshold"] = threshold
    xgb_threshold_results.append(metrics)

xgb_threshold_results = pd.DataFrame(
    xgb_threshold_results
)

xgb_threshold_results

,accuracy,balanced_accuracy,precision,recall,f1,roc_auc,pr_auc,log_loss,threshold
0,0.493042,0.590470,0.302752,0.785714,0.437086,0.687361,0.444308,0.544051,0.10
1,0.612326,0.640984,0.359184,0.698413,0.474394,0.687361,0.444308,0.544051,0.15
2,0.662028,0.637152,0.385417,0.587302,0.465409,0.687361,0.444308,0.544051,0.20
3,0.721670,0.661088,0.453333,0.539683,0.492754,0.687361,0.444308,0.544051,0.25
4,0.755467,0.662498,0.512821,0.476190,0.493827,0.687361,0.444308,0.544051,0.30
5,0.759443,0.628163,0.528736,0.365079,0.431925,0.687361,0.444308,0.544051,0.35
6,0.751491,0.588512,0.507692,0.261905,0.345550,0.687361,0.444308,0.544051,0.40
7,0.757455,0.576639,0.540000,0.214286,0.306818,0.687361,0.444308,0.544051,0.45
8,0.757455,0.558145,0.555556,0.158730,0.246914,0.687361,0.444308,0.544051,0.50


In [54]:
best_threshold_xgb = xgb_threshold_results.loc[
    xgb_threshold_results["f1"].idxmax(),
    "threshold",
]

best_threshold_xgb

np.float64(0.3)

In [55]:
best_threshold_xgb_metrics = xgb_threshold_results.loc[
    xgb_threshold_results["f1"].idxmax()
].to_dict()

best_threshold_xgb_metrics

{'accuracy': 0.7554671968190855,
 'balanced_accuracy': 0.6624984211191107,
 'precision': 0.5128205128205128,
 'recall': 0.47619047619047616,
 'f1': 0.49382716049382713,
 'roc_auc': 0.6873605321881182,
 'pr_auc': 0.44430800057705,
 'log_loss': 0.5440510511398315,
 'threshold': 0.3}

### 6.4 XGBoost Hyperparameter Tuning

We now investigate whether the baseline XGBoost model can be improved through
controlled hyperparameter variation.

As in the Random Forest experiments, hyperparameters are selected using
five-fold time-series cross-validation within the 2010--2020 training period.

The 2021--2022 validation period and the 2023--2026 test period are not used
during hyperparameter selection.

Average Precision (PR-AUC) is used as the primary selection criterion because
the positive class is relatively uncommon. ROC-AUC is reported as a secondary
criterion.

We first investigate the interaction between the number of boosting rounds,
learning rate, and tree depth. Other regularization and subsampling parameters
are kept fixed at their baseline values.

#### 6.4.1 Time-Series Cross-Validation

In [56]:
tscv

TimeSeriesSplit(gap=0, max_train_size=None, n_splits=5, test_size=None)

In [57]:
TimeSeriesSplit(n_splits=5)

TimeSeriesSplit(gap=0, max_train_size=None, n_splits=5, test_size=None)

#### 6.4.2 Boosting Complexity Experiment

In [58]:
n_estimators_values = [100, 200, 400]
learning_rate_values = [0.03, 0.05, 0.1]
max_depth_values = [2, 3, 5]

xgb_complexity_results = []

for n_estimators in n_estimators_values:
    for learning_rate in learning_rate_values:
        for max_depth in max_depth_values:

            model = create_xgboost(
                n_estimators=n_estimators,
                max_depth=max_depth,
                learning_rate=learning_rate,
                subsample=0.8,
                colsample_bytree=0.8,
                min_child_weight=1,
                reg_lambda=1.0,
                reg_alpha=0.0,
                random_state=42,
                n_jobs=-1,
            )

            fold_results = evaluate_classification_model(
                model,
                X_train,
                y_train,
                tscv,
            )

            xgb_complexity_results.append(
                {
                    "n_estimators": n_estimators,
                    "learning_rate": learning_rate,
                    "max_depth": max_depth,
                    "roc_auc_mean": fold_results["roc_auc"].mean(),
                    "roc_auc_std": fold_results["roc_auc"].std(),
                    "pr_auc_mean": fold_results["pr_auc"].mean(),
                    "pr_auc_std": fold_results["pr_auc"].std(),
                }
            )

xgb_complexity_results = pd.DataFrame(
    xgb_complexity_results
)

xgb_complexity_results

,n_estimators,learning_rate,max_depth,roc_auc_mean,roc_auc_std,pr_auc_mean,pr_auc_std
0,100,0.03,2,0.687219,0.057879,0.402350,0.086750
1,100,0.03,3,0.673476,0.063697,0.390856,0.093504
2,100,0.03,5,0.655089,0.066376,0.373200,0.090619
3,100,0.05,2,0.686091,0.060603,0.402393,0.093601
4,100,0.05,3,0.674246,0.068821,0.396923,0.095960
5,100,0.05,5,0.657601,0.072760,0.377878,0.096063
6,100,0.10,2,0.683192,0.073468,0.401313,0.097111
7,100,0.10,3,0.669438,0.067191,0.387978,0.086099
8,100,0.10,5,0.656540,0.063138,0.377880,0.084857
9,200,0.03,2,0.687763,0.061875,0.399969,0.090398


### 6.4.3 Row Subsampling

The first experiment identified relatively shallow trees and a small learning
rate as the most promising configuration.

We now investigate the effect of row subsampling through the `subsample`
parameter while keeping the other parameters fixed.

The candidate values are 0.6, 0.8, and 1.0. The value 1.0 corresponds to
using the complete training sample for each boosting round.

In [59]:
subsample_values = [0.6, 0.8, 1.0]

xgb_subsample_results = []

for subsample in subsample_values:

    model = create_xgboost(
        n_estimators=400,
        max_depth=2,
        learning_rate=0.03,
        subsample=subsample,
        colsample_bytree=0.8,
        min_child_weight=1,
        reg_lambda=1.0,
        reg_alpha=0.0,
        random_state=42,
        n_jobs=-1,
    )

    fold_results = evaluate_classification_model(
        model,
        X_train,
        y_train,
        tscv,
    )

    xgb_subsample_results.append(
        {
            "subsample": subsample,
            "roc_auc_mean": fold_results["roc_auc"].mean(),
            "roc_auc_std": fold_results["roc_auc"].std(),
            "pr_auc_mean": fold_results["pr_auc"].mean(),
            "pr_auc_std": fold_results["pr_auc"].std(),
        }
    )

xgb_subsample_results = pd.DataFrame(
    xgb_subsample_results
)

xgb_subsample_results

,subsample,roc_auc_mean,roc_auc_std,pr_auc_mean,pr_auc_std
0,0.6,0.679108,0.071045,0.397735,0.091542
1,0.8,0.682784,0.070121,0.404234,0.095853
2,1.0,0.675885,0.065679,0.400949,0.090574


### 6.4.4 Current Best Hyperparameters

The first two hyperparameter experiments have identified the following
configuration as the current best according to mean PR-AUC on the
time-series cross-validation folds:

- `n_estimators = 400`
- `learning_rate = 0.03`
- `max_depth = 2`
- `subsample = 0.8`
- `colsample_bytree = 0.8`
- `min_child_weight = 1`
- `reg_lambda = 1.0`
- `reg_alpha = 0.0`

The corresponding mean PR-AUC is approximately 0.404.

This configuration is retained as the current best while additional
regularization parameters are investigated.

In [60]:
best_xgb_params = {
    "n_estimators": 400,
    "learning_rate": 0.03,
    "max_depth": 2,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "min_child_weight": 1,
    "reg_lambda": 1.0,
    "reg_alpha": 0.0,
    "random_state": 42,
    "n_jobs": -1,
}

In [61]:
best_xgb_pr_auc = 0.404234

#### 6.4.5: colsample_bytree

In [62]:
colsample_bytree_values = [0.6, 0.8, 1.0]

xgb_colsample_results = []

for colsample_bytree in colsample_bytree_values:

    params = best_xgb_params.copy()
    params["colsample_bytree"] = colsample_bytree

    model = create_xgboost(**params)

    fold_results = evaluate_classification_model(
        model,
        X_train,
        y_train,
        tscv,
    )

    xgb_colsample_results.append(
        {
            "colsample_bytree": colsample_bytree,
            "roc_auc_mean": fold_results["roc_auc"].mean(),
            "roc_auc_std": fold_results["roc_auc"].std(),
            "pr_auc_mean": fold_results["pr_auc"].mean(),
            "pr_auc_std": fold_results["pr_auc"].std(),
        }
    )

xgb_colsample_results = pd.DataFrame(
    xgb_colsample_results
)

xgb_colsample_results

,colsample_bytree,roc_auc_mean,roc_auc_std,pr_auc_mean,pr_auc_std
0,0.6,0.681202,0.071842,0.400768,0.096388
1,0.8,0.682784,0.070121,0.404234,0.095853
2,1.0,0.679756,0.075945,0.398555,0.094214


In [63]:
best_xgb_params["colsample_bytree"] = 0.8

In [64]:
best_xgb_params

{'n_estimators': 400,
 'learning_rate': 0.03,
 'max_depth': 2,
 'subsample': 0.8,
 'colsample_bytree': 0.8,
 'min_child_weight': 1,
 'reg_lambda': 1.0,
 'reg_alpha': 0.0,
 'random_state': 42,
 'n_jobs': -1}

#### 6.4.6: min_child_weight

In [65]:
min_child_weight_values = [1, 5, 10]

xgb_child_weight_results = []

for min_child_weight in min_child_weight_values:

    params = best_xgb_params.copy()
    params["min_child_weight"] = min_child_weight

    model = create_xgboost(**params)

    fold_results = evaluate_classification_model(
        model,
        X_train,
        y_train,
        tscv,
    )

    xgb_child_weight_results.append(
        {
            "min_child_weight": min_child_weight,
            "roc_auc_mean": fold_results["roc_auc"].mean(),
            "roc_auc_std": fold_results["roc_auc"].std(),
            "pr_auc_mean": fold_results["pr_auc"].mean(),
            "pr_auc_std": fold_results["pr_auc"].std(),
        }
    )

xgb_child_weight_results = pd.DataFrame(
    xgb_child_weight_results
)

xgb_child_weight_results

,min_child_weight,roc_auc_mean,roc_auc_std,pr_auc_mean,pr_auc_std
0,1,0.682784,0.070121,0.404234,0.095853
1,5,0.685009,0.071357,0.410374,0.097274
2,10,0.674948,0.080601,0.395919,0.086763


In [66]:
best_xgb_params["min_child_weight"] = 5

In [67]:
best_xgb_pr_auc = xgb_child_weight_results.loc[
    xgb_child_weight_results["pr_auc_mean"].idxmax(),
    "pr_auc_mean",
]

In [68]:
best_xgb_params

{'n_estimators': 400,
 'learning_rate': 0.03,
 'max_depth': 2,
 'subsample': 0.8,
 'colsample_bytree': 0.8,
 'min_child_weight': 5,
 'reg_lambda': 1.0,
 'reg_alpha': 0.0,
 'random_state': 42,
 'n_jobs': -1}

In [69]:
best_xgb_pr_auc

np.float64(0.4103740489661785)

### 6.4.5 L2 Regularization

The previous experiment showed that increasing `min_child_weight` from 1 to 5
improved the cross-validated PR-AUC.

We now investigate the effect of L2 regularization through the `reg_lambda`
parameter, while keeping the current best configuration fixed.

The candidate values are:

$$
\lambda \in \{0.1, 1.0, 10.0\}.
$$

The model with the highest mean PR-AUC will replace the current best
configuration only if it provides an improvement.

In [70]:
reg_lambda_values = [0.1, 1.0, 10.0]

xgb_lambda_results = []

for reg_lambda in reg_lambda_values:

    params = best_xgb_params.copy()
    params["reg_lambda"] = reg_lambda

    model = create_xgboost(**params)

    fold_results = evaluate_classification_model(
        model,
        X_train,
        y_train,
        tscv,
    )

    xgb_lambda_results.append(
        {
            "reg_lambda": reg_lambda,
            "roc_auc_mean": fold_results["roc_auc"].mean(),
            "roc_auc_std": fold_results["roc_auc"].std(),
            "pr_auc_mean": fold_results["pr_auc"].mean(),
            "pr_auc_std": fold_results["pr_auc"].std(),
        }
    )

xgb_lambda_results = pd.DataFrame(
    xgb_lambda_results
)

xgb_lambda_results

,reg_lambda,roc_auc_mean,roc_auc_std,pr_auc_mean,pr_auc_std
0,0.1,0.685160,0.070970,0.409271,0.097329
1,1.0,0.685009,0.071357,0.410374,0.097274
2,10.0,0.678114,0.069897,0.400105,0.086662


In [71]:
best_lambda_row = xgb_lambda_results.loc[
    xgb_lambda_results["pr_auc_mean"].idxmax()
]

best_lambda = best_lambda_row["reg_lambda"]
best_lambda_pr_auc = best_lambda_row["pr_auc_mean"]

best_lambda, best_lambda_pr_auc

(np.float64(1.0), np.float64(0.4103740489661785))

In [72]:
if best_lambda_pr_auc > best_xgb_pr_auc:
    best_xgb_params["reg_lambda"] = best_lambda
    best_xgb_pr_auc = best_lambda_pr_auc
    lambda_improved = True
else:
    lambda_improved = False

lambda_improved

False

In [73]:
best_xgb_params

{'n_estimators': 400,
 'learning_rate': 0.03,
 'max_depth': 2,
 'subsample': 0.8,
 'colsample_bytree': 0.8,
 'min_child_weight': 5,
 'reg_lambda': 1.0,
 'reg_alpha': 0.0,
 'random_state': 42,
 'n_jobs': -1}

### 6.4.6 L1 Regularization

The L2 regularization experiment did not improve the current best
configuration.

We now investigate L1 regularization through the `reg_alpha` parameter while
keeping all other parameters fixed at their current best values.

The candidate values are:

$$
\alpha \in \{0.0, 0.1, 1.0\}.
$$

The configuration with the highest mean PR-AUC will replace the current best
configuration only if it produces a strict improvement.

In [74]:
reg_alpha_values = [0.0, 0.1, 1.0]

xgb_alpha_results = []

for reg_alpha in reg_alpha_values:

    params = best_xgb_params.copy()
    params["reg_alpha"] = reg_alpha

    model = create_xgboost(**params)

    fold_results = evaluate_classification_model(
        model,
        X_train,
        y_train,
        tscv,
    )

    xgb_alpha_results.append(
        {
            "reg_alpha": reg_alpha,
            "roc_auc_mean": fold_results["roc_auc"].mean(),
            "roc_auc_std": fold_results["roc_auc"].std(),
            "pr_auc_mean": fold_results["pr_auc"].mean(),
            "pr_auc_std": fold_results["pr_auc"].std(),
        }
    )

xgb_alpha_results = pd.DataFrame(
    xgb_alpha_results
)

xgb_alpha_results

,reg_alpha,roc_auc_mean,roc_auc_std,pr_auc_mean,pr_auc_std
0,0.0,0.685009,0.071357,0.410374,0.097274
1,0.1,0.684453,0.070445,0.409153,0.094528
2,1.0,0.682817,0.072529,0.408741,0.096723


In [75]:
best_alpha_row = xgb_alpha_results.loc[
    xgb_alpha_results["pr_auc_mean"].idxmax()
]

best_alpha = best_alpha_row["reg_alpha"]
best_alpha_pr_auc = best_alpha_row["pr_auc_mean"]

best_alpha, best_alpha_pr_auc

(np.float64(0.0), np.float64(0.4103740489661785))

check if L1 regularization provides an improvement

In [76]:
if best_alpha_pr_auc > best_xgb_pr_auc:
    best_xgb_params["reg_alpha"] = best_alpha
    best_xgb_pr_auc = best_alpha_pr_auc
    alpha_improved = True
else:
    alpha_improved = False

alpha_improved

False

In [77]:
# best hyperparameters should be same 
best_xgb_params

{'n_estimators': 400,
 'learning_rate': 0.03,
 'max_depth': 2,
 'subsample': 0.8,
 'colsample_bytree': 0.8,
 'min_child_weight': 5,
 'reg_lambda': 1.0,
 'reg_alpha': 0.0,
 'random_state': 42,
 'n_jobs': -1}

In [78]:
best_xgb_pr_auc

np.float64(0.4103740489661785)

### 6.5 Selected XGBoost Configuration

The hyperparameter experiments were performed using five-fold time-series
cross-validation within the 2010--2020 training period.

The final configuration selected by the experiments is stored in
`best_xgb_params`. The selection criterion is mean PR-AUC, with ROC-AUC used
as a secondary metric.

The selected configuration is now constructed directly from this parameter
dictionary rather than re-entering the hyperparameter values manually.

The 2021--2022 validation period will now be used to evaluate this selected
configuration and to select its classification threshold.

In [79]:
selected_xgb = create_xgboost(
    **best_xgb_params
)

In [80]:
# fit on the training period only
selected_xgb.fit(
    X_train,
    y_train,
)

,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",'logloss'
,feature_types feature_types: typing.Optional[typing.Sequence[str]].. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [81]:
# verify fitting -> True
hasattr(selected_xgb, "n_features_in_")

True

In [82]:
selected_xgb.get_booster().num_boosted_rounds()

400

#### 6.5.1: Validation Prediction

In [83]:
y_validation_proba_selected_xgb = selected_xgb.predict_proba(
    X_validation
)[:, 1]

In [84]:
# caclulate default 0.5 prediction
y_validation_pred_selected_xgb = selected_xgb.predict(
    X_validation
)

In [85]:
# evaluate them
selected_xgb_validation_default_metrics = classification_metrics(
    y_validation,
    y_validation_pred_selected_xgb,
    y_validation_proba_selected_xgb,
)

selected_xgb_validation_default_metrics

{'accuracy': 0.757455268389662,
 'balanced_accuracy': 0.5475769441286682,
 'precision': 0.5714285714285714,
 'recall': 0.12698412698412698,
 'f1': 0.2077922077922078,
 'roc_auc': 0.6797398004294557,
 'pr_auc': 0.47069465215879513,
 'log_loss': 0.5430222153663635}

So tuning did not improve every metric. In particular, ROC-AUC decreased slightly and default-threshold F1 decreased. But PR-AUC increased from 0.444 to 0.471, which is exactly the metric we chose as the primary hyperparameter-selection criterion.
That's an important result to report rather than trying to make the tuning look uniformly better.

#### 6.5.2 Threshold Selection

In [86]:
thresholds_selected_xgb = [
    0.10,
    0.15,
    0.20,
    0.25,
    0.30,
    0.35,
    0.40,
    0.45,
    0.50,
]

selected_xgb_threshold_results = []

for threshold in thresholds_selected_xgb:

    y_pred_threshold = (
        y_validation_proba_selected_xgb >= threshold
    ).astype(int)

    metrics = classification_metrics(
        y_validation,
        y_pred_threshold,
        y_validation_proba_selected_xgb,
    )

    metrics["threshold"] = threshold
    selected_xgb_threshold_results.append(metrics)

selected_xgb_threshold_results = pd.DataFrame(
    selected_xgb_threshold_results
)

selected_xgb_threshold_results

,accuracy,balanced_accuracy,precision,recall,f1,roc_auc,pr_auc,log_loss,threshold
0,0.471173,0.570597,0.290419,0.769841,0.421739,0.67974,0.470695,0.543022,0.10
1,0.590457,0.623753,0.342520,0.690476,0.457895,0.67974,0.470695,0.543022,0.15
2,0.683897,0.657025,0.410811,0.603175,0.488746,0.67974,0.470695,0.543022,0.20
3,0.705765,0.639910,0.426667,0.507937,0.463768,0.67974,0.470695,0.543022,0.25
4,0.751491,0.649278,0.504505,0.444444,0.472574,0.67974,0.470695,0.543022,0.30
5,0.769384,0.634794,0.560976,0.365079,0.442308,0.67974,0.470695,0.543022,0.35
6,0.767396,0.599122,0.578947,0.261905,0.360656,0.67974,0.470695,0.543022,0.40
7,0.765408,0.576660,0.595238,0.198413,0.297619,0.67974,0.470695,0.543022,0.45
8,0.757455,0.547577,0.571429,0.126984,0.207792,0.67974,0.470695,0.543022,0.50


In [87]:
best_threshold_selected_xgb_row = (
    selected_xgb_threshold_results.loc[
        selected_xgb_threshold_results["f1"].idxmax()
    ]
)

best_threshold_selected_xgb = (
    best_threshold_selected_xgb_row["threshold"]
)

best_threshold_selected_xgb

np.float64(0.2)

In [88]:
best_threshold_selected_xgb_metrics = (
    best_threshold_selected_xgb_row.to_dict()
)

best_threshold_selected_xgb_metrics

{'accuracy': 0.68389662027833,
 'balanced_accuracy': 0.657024967369795,
 'precision': 0.41081081081081083,
 'recall': 0.6031746031746031,
 'f1': 0.4887459807073955,
 'roc_auc': 0.6797398004294557,
 'pr_auc': 0.47069465215879513,
 'log_loss': 0.5430222153663635,
 'threshold': 0.2}

### 6.5.3 Final Validation Evaluation

The XGBoost hyperparameters have been selected using time-series
cross-validation on the 2010--2020 training period.

The classification threshold has subsequently been selected on the
2021--2022 validation period. Among the candidate thresholds examined,
`0.20` produced the highest F1 score.

We now record the final validation performance of the selected XGBoost
configuration at this threshold.

No further model or threshold selection is performed in this section.

In [89]:
y_validation_pred_selected_xgb = (
    y_validation_proba_selected_xgb
    >= best_threshold_selected_xgb
).astype(int)

In [90]:
selected_xgb_validation_metrics = classification_metrics(
    y_validation,
    y_validation_pred_selected_xgb,
    y_validation_proba_selected_xgb,
)

selected_xgb_validation_metrics

{'accuracy': 0.68389662027833,
 'balanced_accuracy': 0.657024967369795,
 'precision': 0.41081081081081083,
 'recall': 0.6031746031746031,
 'f1': 0.4887459807073955,
 'roc_auc': 0.6797398004294557,
 'pr_auc': 0.47069465215879513,
 'log_loss': 0.5430222153663635}

### 6.6 Final Test Evaluation

The XGBoost hyperparameters were selected using time-series cross-validation
on the 2010--2020 training period, and the classification threshold was
selected using the 2021--2022 validation period.

We now freeze both choices.

The training and validation observations are combined to form the final
development dataset. The selected XGBoost model is then refitted on this
combined 2010--2022 dataset and evaluated once on the untouched 2023--2026
test period.

The test set is not used for any model or threshold selection.

#### 6.6.1 Combine the Development Data

In [91]:
development_data = pd.concat(
    [
        train_data,
        validation_data,
    ]
)

X_development = development_data[FEATURE_COLUMNS]
y_development = development_data["target"]

In [92]:
# verify it
print(
    "Development:",
    X_development.shape,
    y_development.shape,
)

print(
    "Test:",
    X_test.shape,
    y_test.shape,
)

Development: (3212, 27) (3212,)
Test: (919, 27) (919,)


#### 6.6.2 Refit the Selected XGBoost

In [93]:
final_xgb = create_xgboost(
    **best_xgb_params
)

In [94]:
final_xgb.fit(
    X_development,
    y_development,
)

,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",'logloss'
,feature_types feature_types: typing.Optional[typing.Sequence[str]].. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [95]:
# verify the fitting -> True
hasattr(final_xgb, "n_features_in_")

True

In [96]:
final_xgb.get_booster().num_boosted_rounds()

400

#### 6.6.3 Generate Test Predictions

In [97]:
y_test_proba_xgb = final_xgb.predict_proba(
    X_test
)[:, 1]

In [98]:
# apply the frozen validation threshold
y_test_pred_xgb = (
    y_test_proba_xgb >= best_threshold_selected_xgb
).astype(int)

In [99]:
# evaluate
final_xgb_test_metrics = classification_metrics(
    y_test,
    y_test_pred_xgb,
    y_test_proba_xgb,
)

final_xgb_test_metrics

{'accuracy': 0.5712731229597389,
 'balanced_accuracy': 0.6636127355425601,
 'precision': 0.3023255813953488,
 'recall': 0.8210526315789474,
 'f1': 0.44192634560906513,
 'roc_auc': 0.7359468630423796,
 'pr_auc': 0.3901385814270502,
 'log_loss': 0.46058914065361023}

# 7. Tree-Model Conclusions

The Stage-0 tree-model experiments compared Random Forest and XGBoost for
the binary volatility-prediction problem.

Both models were evaluated using chronological data splits. Hyperparameters
were selected using time-series cross-validation on the 2010--2020 training
period, classification thresholds were selected using the 2021--2022
validation period, and the final models were refitted on the combined
2010--2022 development period before being evaluated once on the untouched
2023--2026 test period.

The final test results are summarized below.

| Metric | Random Forest | XGBoost |
|---|---:|---:|
| Accuracy | 0.589 | 0.571 |
| Balanced Accuracy | 0.677 | 0.664 |
| Precision | 0.313 | 0.302 |
| Recall | 0.826 | 0.821 |
| F1 | 0.454 | 0.442 |
| ROC-AUC | 0.756 | 0.736 |
| PR-AUC | 0.422 | 0.390 |
| Log Loss | 0.454 | 0.461 |

The Random Forest provides the stronger final test performance across all of
the principal metrics considered here. In particular, it achieves higher
ROC-AUC, PR-AUC, and F1 than XGBoost.

The relatively high recall of both models comes from using a classification
threshold below the default value of 0.5. The threshold was selected using
the validation period and then kept fixed for the final test evaluation.

The results indicate that nonlinear tree-based models can extract useful
information from the Stage-0 feature set. However, the difference between
Random Forest and XGBoost is not large enough to suggest that a substantially
more complex tree-based model solves the prediction problem.

The results also reinforce an important observation from the internship
project: in this relatively low-to-moderate data regime, tree-based models
provide a strong baseline for volatility prediction.

The Random Forest will therefore serve as an important reference model for
the subsequent stages of the project. Stage 1 will investigate whether a
one-dimensional convolutional neural network can extract additional
information directly from temporal sequences, while later stages will
investigate data augmentation using symmetry, distribution, topology, and
geometric structure.

## Save Models

In [100]:
import joblib

MODELS_DIR = PROJECT_ROOT / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Save the final Random Forest model.
joblib.dump(
    final_random_forest,
    MODELS_DIR / "stage0_random_forest.joblib",
)

# Save the final XGBoost model.
joblib.dump(
    final_xgb,
    MODELS_DIR / "stage0_xgboost.joblib",
)

['/Users/amitkumar/Desktop/Computing/Projects/Finance/market-volatility/models/stage0_xgboost.joblib']